In [1]:
import cv2
import mediapipe as mp
import time
import numpy as np

In [2]:
import keyboard

In [3]:
import PythonToFLStudioBridge_mod as PTSB

Listening on loopMIDI Port 1 1 for control 2...
Timeout reached, no message received.
0
end


In [4]:
class HandDetection:
    def __init__(self, mode=False, maxHands=2, detectionCon=0.5, trackCon=0.5):
        self.mode = mode
        self.maxHands = maxHands
        self.detectionCon = detectionCon
        self.trackCon = trackCon
        
        self.mpHands = mp.solutions.hands
        self.hands = self.mpHands.Hands(static_image_mode=self.mode, 
                                        max_num_hands=self.maxHands,
                                        min_detection_confidence=self.detectionCon, 
                                        min_tracking_confidence=self.trackCon)
        self.mpDraw = mp.solutions.drawing_utils

    def findHands(self, img, draw=True):
        imgRGB = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        self.results = self.hands.process(imgRGB)
        
        if self.results.multi_hand_landmarks:
            for handLms in self.results.multi_hand_landmarks:
                if draw: self.mpDraw.draw_landmarks(img, handLms, self.mpHands.HAND_CONNECTIONS)
        return img

    def findPosition(self, img, handNo=0, draw=True, axis=False):
        lmList = []
        if self.results.multi_hand_landmarks:
            myHand = self.results.multi_hand_landmarks[handNo]
            
            for id, lm in enumerate(myHand.landmark):
                h, w, c = img.shape
                cx, cy = int(lm.x * w), int(lm.y * h)
                lmList.append([id, cx, cy])
                if draw:
                    cv2.circle(img, (cx, cy), 25, (255, 255, 255), cv2.FILLED)
        return lmList
    
    def drawAxis(self, img, axis=False):
        if axis:
            height, width, _ = img.shape
            center_x, center_y = width // 2, height // 2
            cv2.line(img, (center_x, 0), (center_x, height), (0, 255, 0), 2)  # Y-axis
            cv2.line(img, (0, center_y), (width, center_y), (255, 0, 0), 2)  # X-axis
            cv2.putText(img, f"(0, {center_y})", (10, center_y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1, cv2.LINE_AA)
            cv2.putText(img, f"({width}, {center_y})", (width - 150, center_y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1, cv2.LINE_AA)
            cv2.putText(img, f"({center_x}, 0)", (center_x + 10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1, cv2.LINE_AA)
            cv2.putText(img, f"({center_x}, {height})", (center_x + 10, height - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1, cv2.LINE_AA)


In [5]:
class HandControl:
    def __init__(self, detector: HandDetection):
        self.detector = detector

    def indexThumbDistance(self, img, draw=False, draw_interval=False):
        mid = []
        if self.detector.results and self.detector.results.multi_hand_landmarks:
            for hand in self.detector.results.multi_hand_landmarks:
                if len(hand.landmark) > 7:
                    index = hand.landmark[8]
                    thumb = hand.landmark[4]
                    
                    h, w, c = img.shape
                    index_x, index_y = int(index.x * w), int(index.y * h)
                    thumb_x, thumb_y = int(thumb.x * w), int(thumb.y * h)
                    
                    middle_x = (index_x + thumb_x) // 2
                    middle_y = (index_y + thumb_y) // 2
                    
                    if draw: 
                        self.detector.findHands(img, draw=True)
                        self.detector.findPosition(img, draw=True, axis=False)
                        self.detector.drawAxis(img, axis=True)
                        self.detector.PointConnect(img, (thumb_x, thumb_y), (index_x, index_y), (middle_x, middle_y))
                    
                    mid.append((middle_x, middle_y))
            if len(mid) == 2 and draw_interval: 
                cv2.line(img, mid[0], mid[1], (255, 255, 255), 2)
                length = self.detector.Length(mid)
                PTSB.SendCC(percent=length / 356, ctrl=2)
        return mid

    def handDirection(self, img, draw=False):
        if self.detector.results and self.detector.results.multi_hand_landmarks:
            for hand in self.detector.results.multi_hand_landmarks:
                if len(hand.landmark) > 7:
                    index = hand.landmark[8]
                    thumb = hand.landmark[4]
                    
                    h, w, c = img.shape
                    index_x, index_y = int(index.x * w), int(index.y * h)
                    thumb_x, thumb_y = int(thumb.x * w), int(thumb.y * h)
                    
                    if thumb_x < index_x:
                        return True
                    else:
                        return False
        return None


In [6]:
class SignalSender:
    @staticmethod
    def sendControlChange(lmList, counter, draw=False):
        if len(lmList) > 2 and counter % 5 == 0:
            percents = PTSB.data_writer(data=lmList[8][1:])
            PTSB.SendCC(percent=percents[1], ctrl=1)
            counter += 1
        return counter

In [8]:
pTime = 0
cTime = 0
cap = cv2.VideoCapture(0)
detector = HandDetection()
controller = HandControl(detector)
signal_sender = SignalSender()
draw = False
draw_interval = False
axis = False
counter = 0

while True:
    success, img = cap.read()
    img = detector.findHands(img)
    lmList = detector.findPosition(img, draw=False, axis=axis)
    middle_points = controller.indexThumbDistance(img, draw=draw, draw_interval=draw_interval)
    counter = signal_sender.sendControlChange(lmList, counter, draw=False)
    
    
    
    # CC_value = PTSB.getCC(2)
    # cv2.putText(img, str(CC_value), (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 3)
    
    
    
    cTime = time.time()
    fps = 1 / (cTime - pTime)
    pTime = cTime
    
    cv2.putText(img, str(int(fps)), (10, 10), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 3)
    cv2.putText(img, str(middle_points), (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    cv2.imshow("Image", img)
    
    
    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'): break
    if key == ord('d'): draw = not draw
    if key == ord('i'): draw_interval = not draw_interval
    if key == ord('a'): axis = not axis
    
cap.release()
cv2.destroyAllWindows()

Data saved to C:\Users\Niitro_musics\Documents\Image-Line\FL Studio\Settings\Hardware\a\script.py
Sent CC message: control_change channel=1 control=1 value=75 time=0


In [ ]:
# while True:
#     value = PTSB.getCC(2,channel=0)

Listening on loopMIDI Port 1 1 for control 2...
Timeout reached, no message received.
Listening on loopMIDI Port 1 1 for control 2...
Timeout reached, no message received.
Listening on loopMIDI Port 1 1 for control 2...
